# Climate Change + Building Code Sensitivity Analysis (Wind + Flood)

**Research Question**: How much combined wind+flood building code improvement is needed to offset SSP245 2050 climate change impacts?

**Approach**:
1. Compute climate delta: median across 5 GCMs of (SSP245 - Historical)
2. Apply to ERA5 baseline for each combined building code level
3. Evaluate using metrics calculated AFTER building codes applied

**Building Code Levels** (combined wind+flood reductions):
- **0%**: w00f00 - No building codes (baseline climate impact)
- **20/13%**: w20f13 - 20% wind + 13% flood reduction
- **30/20%**: w30f20 - 30% wind + 20% flood reduction (current strong codes)
- **40/27%**: w40f27 - 40% wind + 27% flood reduction
- **50/33%**: w50f33 - 50% wind + 33% flood reduction  
- **60/40%**: w60f40 - 60% wind + 40% flood reduction
- **70/47%**: w70f47 - 70% wind + 47% flood reduction
- **80/53%**: w80f53 - 80% wind + 53% flood reduction
- **90/60%**: w90f60 - 90% wind + 60% flood reduction
- **100/67%**: w100f67 - 100% wind + 67% flood reduction (3:2 ratio max)
- **100/70%**: w110f70 - 100% wind + 70% flood reduction
- **100/80%**: w120f80 - 100% wind + 80% flood reduction
- **100/90%**: w130f90 - 100% wind + 90% flood reduction (maximum feasible)

**Key Metrics**:
- **Total Economic Burden**: Insured + Underinsured + Uninsured losses
- **Defaults**: Private insurance company failures
- **Total Public Burden**: Sum of FIGA + Citizens + NFIP + FHCF deficits

**Note**: This analysis uses combined wind+flood building code effectiveness based on empirical damage ratios.

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

results_dir = Path('../results/mc_runs')

## 1. Load Data

Load all necessary simulation runs:
- GCM historical baselines (20thcal)
- GCM SSP245 with combined wind+flood building codes (w00f00 through w60f40)
- ERA5 baseline

In [ ]:
# Define building code levels (wind%, flood%)
building_code_levels = {
    'w00f00': (0, 0),
    'w20f13': (20, 13),
    'w30f20': (30, 20),
    'w40f27': (40, 27),
    'w50f33': (50, 33),
    'w60f40': (60, 40),
    'w70f47': (70, 47),
    'w80f53': (80, 53),
    'w90f60': (90, 60),
    'w100f67': (100, 67),
    'w110f70': (100, 70),
    'w120f80': (100, 80),
    'w130f90': (100, 90),
}

# GCM models to analyze
gcms = ['canesm', 'cnrm6', 'ecearth6', 'ipsl6', 'miroc6']

print("Loading GCM historical baselines (20thcal)...")
gcm_baselines = {}
for gcm in gcms:
    matching_dirs = sorted(results_dir.glob(f'emanuel_{gcm}_20thcal_baseline_*'))
    for run_dir in reversed(matching_dirs):  # Get most recent
        if (run_dir / 'iterations.csv').exists():
            gcm_baselines[gcm] = pd.read_csv(run_dir / 'iterations.csv')
            print(f"  ✓ {gcm} historical: {len(gcm_baselines[gcm])} iterations from {run_dir.name}")
            break

print(f"\nLoading GCM SSP245 runs with building codes...")
gcm_futures_with_codes = {}
for gcm in gcms:
    gcm_futures_with_codes[gcm] = {}
    for bc_label, (wind_pct, flood_pct) in building_code_levels.items():
        pattern = f'emanuel_{gcm}_ssp245cal_buildingcode_{bc_label}_*'
        matching_dirs = sorted(results_dir.glob(pattern))
        if matching_dirs:
            run_dir = matching_dirs[-1]  # Get most recent
            if (run_dir / 'iterations.csv').exists():
                gcm_futures_with_codes[gcm][bc_label] = pd.read_csv(run_dir / 'iterations.csv')
                print(f"  ✓ {gcm} {bc_label} (w{wind_pct}%/f{flood_pct}%): {len(gcm_futures_with_codes[gcm][bc_label])} iterations from {run_dir.name}")

print(f"\nLoading ERA5 baseline...")
era5_baseline = None
matching_dirs = sorted(results_dir.glob('emanuel_era5_baseline_*'))
for run_dir in reversed(matching_dirs):  # Get most recent
    if (run_dir / 'iterations.csv').exists():
        era5_baseline = pd.read_csv(run_dir / 'iterations.csv')
        print(f"  ✓ ERA5 baseline: {len(era5_baseline)} iterations from {run_dir.name}")
        break

print(f"\n{'='*60}")
print(f"Summary:")
print(f"  GCM models: {len(gcm_baselines)}")
print(f"  Building code levels: {len(building_code_levels)}")
print(f"  Total GCM×BC runs: {sum(len(runs) for runs in gcm_futures_with_codes.values())}")
print(f"{'='*60}")

## 2. Define Metric Computation

Compute metrics that properly reflect building code reductions (calculated AFTER building codes applied):
- Individual components: insured, underinsured, uninsured
- Composite metrics: total economic burden, total public burden

In [ ]:
def compute_metrics(df):
    """Compute metrics that properly reflect building code reductions."""
    df_valid = df[df['scenario'] != 'error'].copy()
    if len(df_valid) == 0:
        return None
    
    metrics = {}
    
    # Economic burden components (all decrease with building codes)
    metrics['wind_insured_private_usd'] = df_valid['wind_insured_private_usd'].mean()
    metrics['wind_insured_citizens_usd'] = df_valid['wind_insured_citizens_usd'].mean()
    metrics['flood_insured_capped_usd'] = df_valid['flood_insured_capped_usd'].mean()
    metrics['wind_underinsured_usd'] = df_valid['wind_underinsured_usd'].mean()
    metrics['wind_uninsured_usd'] = df_valid['wind_uninsured_usd'].mean()
    
    # Private market stress
    metrics['defaults_post'] = df_valid['defaults_post'].mean()
    
    # Public sector deficits
    metrics['figa_residual_deficit_usd'] = df_valid['figa_residual_deficit_usd'].mean()
    metrics['citizens_residual_deficit_usd'] = df_valid['citizens_residual_deficit_usd'].mean()
    metrics['nfip_borrowed_usd'] = df_valid['nfip_borrowed_usd'].mean()
    metrics['fhcf_shortfall_usd'] = df_valid['fhcf_shortfall_usd'].mean()
    
    # Composite: Total insured loss
    metrics['total_insured_usd'] = (
        metrics['wind_insured_private_usd'] +
        metrics['wind_insured_citizens_usd'] +
        metrics['flood_insured_capped_usd']
    )
    
    # Composite: Total protected loss (insured + underinsured)
    metrics['total_protected_usd'] = (
        metrics['total_insured_usd'] +
        metrics['wind_underinsured_usd']
    )
    
    # Composite: Total economic burden (all losses)
    metrics['total_economic_burden_usd'] = (
        metrics['total_protected_usd'] +
        metrics['wind_uninsured_usd']
    )
    
    # Composite: Total public burden (all government deficits)
    metrics['total_public_burden_usd'] = (
        metrics['figa_residual_deficit_usd'] +
        metrics['citizens_residual_deficit_usd'] +
        metrics['nfip_borrowed_usd'] +
        metrics['fhcf_shortfall_usd']
    )
    
    return metrics

# Test on ERA5
if era5_baseline is not None:
    era5_metrics = compute_metrics(era5_baseline)
    print("ERA5 baseline metrics (mean across iterations):")
    print(f"  Total Economic Burden: ${era5_metrics['total_economic_burden_usd']/1e9:.2f}B")
    print(f"  Defaults: {era5_metrics['defaults_post']:.2f}")
    print(f"  Total Public Burden: ${era5_metrics['total_public_burden_usd']/1e9:.2f}B")

## 3. Compute Climate Delta (GCM Ensemble)

For each building code level, compute median climate change impact across 5 GCMs:
- Climate delta = SSP245 (with building code) - Historical baseline
- Median across GCMs to get ensemble estimate

In [ ]:
# Process each building code level
results = []

for bc_label, (wind_pct, flood_pct) in building_code_levels.items():
    # Collect deltas from all GCMs for this building code level
    gcm_deltas_bc = []
    
    for gcm in gcms:
        if bc_label not in gcm_futures_with_codes[gcm] or gcm not in gcm_baselines:
            continue
        
        metrics_bc = compute_metrics(gcm_futures_with_codes[gcm][bc_label])
        metrics_hist = compute_metrics(gcm_baselines[gcm])
        
        if metrics_bc and metrics_hist:
            delta = {k: metrics_bc[k] - metrics_hist[k] for k in metrics_bc.keys()}
            gcm_deltas_bc.append(delta)
            print(f"{gcm} {bc_label}: Δ Economic = ${delta['total_economic_burden_usd']/1e9:+.2f}B")
    
    # Compute median AND p10/p90 climate delta across GCMs for this building code level
    if gcm_deltas_bc:
        climate_bc_delta_median = {}
        climate_bc_delta_p10 = {}
        climate_bc_delta_p90 = {}
        
        for metric in gcm_deltas_bc[0].keys():
            values = [d[metric] for d in gcm_deltas_bc]
            climate_bc_delta_median[metric] = np.median(values)
            climate_bc_delta_p10[metric] = np.percentile(values, 10)
            climate_bc_delta_p90[metric] = np.percentile(values, 90)
        
        # Apply climate delta to ERA5 baseline (median, p10, p90)
        era5_future_bc_median = {k: era5_metrics[k] + climate_bc_delta_median[k] for k in era5_metrics.keys()}
        era5_future_bc_p10 = {k: era5_metrics[k] + climate_bc_delta_p10[k] for k in era5_metrics.keys()}
        era5_future_bc_p90 = {k: era5_metrics[k] + climate_bc_delta_p90[k] for k in era5_metrics.keys()}
        
        # Store results
        combined_pct = (wind_pct + flood_pct) / 2
        results.append({
            'building_code_label': bc_label,
            'wind_pct': wind_pct,
            'flood_pct': flood_pct,
            'combined_pct': combined_pct,
            # Median values
            'economic_burden_b': era5_future_bc_median['total_economic_burden_usd'] / 1e9,
            'defaults': era5_future_bc_median['defaults_post'],
            'public_burden_b': era5_future_bc_median['total_public_burden_usd'] / 1e9,
            # P10 (lower bound)
            'economic_burden_b_p10': era5_future_bc_p10['total_economic_burden_usd'] / 1e9,
            'defaults_p10': era5_future_bc_p10['defaults_post'],
            'public_burden_b_p10': era5_future_bc_p10['total_public_burden_usd'] / 1e9,
            # P90 (upper bound)
            'economic_burden_b_p90': era5_future_bc_p90['total_economic_burden_usd'] / 1e9,
            'defaults_p90': era5_future_bc_p90['defaults_post'],
            'public_burden_b_p90': era5_future_bc_p90['total_public_burden_usd'] / 1e9,
        })
        
        print(f"  → Climate delta for {bc_label} (median [p10-p90]):")
        print(f"     Economic Burden: ${climate_bc_delta_median['total_economic_burden_usd']/1e9:+.2f}B [{climate_bc_delta_p10['total_economic_burden_usd']/1e9:+.2f}, {climate_bc_delta_p90['total_economic_burden_usd']/1e9:+.2f}]")
        print(f"     Defaults: {climate_bc_delta_median['defaults_post']:+.2f} [{climate_bc_delta_p10['defaults_post']:+.2f}, {climate_bc_delta_p90['defaults_post']:+.2f}]")
        print(f"     Public Burden: ${climate_bc_delta_median['total_public_burden_usd']/1e9:+.2f}B [{climate_bc_delta_p10['total_public_burden_usd']/1e9:+.2f}, {climate_bc_delta_p90['total_public_burden_usd']/1e9:+.2f}]")
        print()

df_results = pd.DataFrame(results).sort_values('combined_pct')

print("\nResults Summary (Median [P10-P90] across GCMs):")
print(df_results[['building_code_label', 'combined_pct', 'economic_burden_b', 'defaults', 'public_burden_b']].to_string(index=False))

## 4. Apply Climate Delta to ERA5 Baseline

Add GCM-ensemble climate delta to ERA5 baseline for each building code level:
- This gives us climate-adjusted ERA5 results for each building code scenario
- Allows us to see how much building codes offset climate change

In [ ]:
# ERA5 baseline values for comparison
era5_baseline_values = {
    'economic_burden_b': era5_metrics['total_economic_burden_usd'] / 1e9,
    'defaults': era5_metrics['defaults_post'],
    'public_burden_b': era5_metrics['total_public_burden_usd'] / 1e9,
}

print("ERA5 Baseline (Current Climate):")
print(f"  Economic burden: ${era5_baseline_values['economic_burden_b']:.2f}B")
print(f"  Defaults: {era5_baseline_values['defaults']:.2f}")
print(f"  Public burden: ${era5_baseline_values['public_burden_b']:.2f}B\n")

print("Climate-adjusted ERA5 with building codes:")
for _, row in df_results.iterrows():
    print(f"\n{row['building_code_label']} (w{row['wind_pct']:.0f}%/f{row['flood_pct']:.0f}%, combined {row['combined_pct']:.1f}%):")
    print(f"  Economic Burden: ${row['economic_burden_b']:.2f}B")
    print(f"  Defaults: {row['defaults']:.2f}")
    print(f"  Public Burden: ${row['public_burden_b']:.2f}B")

## 5. Visualize Building Code Effectiveness

Create plots showing:
1. How each metric changes with building code level
2. Comparison to current climate baseline
3. Identification of building code level needed to offset climate change

In [ ]:
# Already have df_results from previous cell
print("Data prepared for visualization:")
print("\nMedian values:")
print(df_results[['building_code_label', 'combined_pct', 'economic_burden_b', 'defaults', 'public_burden_b']])
print("\nUncertainty bounds (P10-P90 across GCMs):")
print(df_results[['building_code_label', 'economic_burden_b_p10', 'economic_burden_b_p90', 
                   'defaults_p10', 'defaults_p90', 'public_burden_b_p10', 'public_burden_b_p90']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11,3), sharex=True, facecolor='white')

# Panel A: Total economic burden
ax = axes[0]
ax.set_facecolor('white')
# Median line
ax.plot(df_results['combined_pct'], df_results['economic_burden_b'], 
        'o-', color='k', alpha=0.85, linewidth=1.5, markersize=4, label='SSP245 2050 (median)', zorder=3)
# P10-P90 uncertainty band
ax.fill_between(df_results['combined_pct'], 
                df_results['economic_burden_b_p10'], 
                df_results['economic_burden_b_p90'],
                color='k', alpha=0.15, linewidth=0, label='P10-P90 range', zorder=1)
# Baseline
ax.axhline(era5_baseline_values['economic_burden_b'], color='k', 
           linestyle='--', linewidth=1.0, alpha=0.6, label='ERA5', zorder=2)
ax.set_ylabel('Annual total loss (billion USD)', fontsize=12)
ax.text(-0.1, 1.08, 'a', transform=ax.transAxes, fontsize=12, fontweight='bold', va='top')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.tick_params(direction='out', length=4, width=1.0, labelsize=10)
ax.grid(False)
ax.set_xlim(-5, 105)

# Panel B: Defaults
ax = axes[1]
ax.set_facecolor('white')
# Median line
ax.plot(df_results['combined_pct'], df_results['defaults'], 
        'o-', color='k', alpha=0.85, linewidth=1.5, markersize=4, zorder=3)
# P10-P90 uncertainty band
ax.fill_between(df_results['combined_pct'], 
                df_results['defaults_p10'], 
                df_results['defaults_p90'],
                color='k', alpha=0.15, linewidth=0, zorder=1)
# Baseline
ax.axhline(era5_baseline_values['defaults'], color='k', 
           linestyle='--', linewidth=1.0, alpha=0.6, zorder=2)
ax.set_ylabel('Annual defaults', fontsize=12)
ax.text(-0.1, 1.08, 'b', transform=ax.transAxes, fontsize=12, fontweight='bold', va='top')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.tick_params(direction='out', length=4, width=1.0, labelsize=10)
ax.grid(False)
ax.set_xlim(-5, 105)

# Panel C: Total public burden
ax = axes[2]
ax.set_facecolor('white')
# Median line
ax.plot(df_results['combined_pct'], df_results['public_burden_b'], 
        'o-', color='k', alpha=0.85, linewidth=1.5, markersize=4, zorder=3)
# P10-P90 uncertainty band
ax.fill_between(df_results['combined_pct'], 
                df_results['public_burden_b_p10'], 
                df_results['public_burden_b_p90'],
                color='k', alpha=0.15, linewidth=0, zorder=1)
# Baseline
ax.axhline(era5_baseline_values['public_burden_b'], color='k', 
           linestyle='--', linewidth=1.0, alpha=0.6, zorder=2)
ax.set_ylabel('Annual public burden (billion USD)', fontsize=12)
ax.text(-0.1, 1.08, 'c', transform=ax.transAxes, fontsize=12, fontweight='bold', va='top')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.tick_params(direction='out', length=4, width=1.0, labelsize=10)
ax.grid(False)
ax.set_xlim(-5, 105)

# Shared x-axis label
fig.text(0.48, -0.05, 'Building code loss reduction (%)', ha='center', fontsize=12)

# Shared legend to the right
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.85, 0.5), 
           frameon=False, fontsize=10)

plt.tight_layout()
plt.subplots_adjust(right=0.88)

# Save figure
out_file = Path('../results/figures/fig_climate_buildingcode_sensitivity.png')
out_file.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✓ Figure saved: {out_file}")

## 6. Determine Building Code Level Needed

Calculate what building code level is needed to offset climate change and return to ERA5 baseline levels for each metric.

In [ ]:
from scipy.interpolate import interp1d

# Get baseline values
baseline_economic = era5_baseline_values['economic_burden_b']
baseline_defaults = era5_baseline_values['defaults']
baseline_public = era5_baseline_values['public_burden_b']

# Create interpolation functions (median values)
f_econ = interp1d(df_results['economic_burden_b'], df_results['combined_pct'], 
                   kind='linear', fill_value='extrapolate')
f_defaults = interp1d(df_results['defaults'], df_results['combined_pct'], 
                      kind='linear', fill_value='extrapolate')
f_public = interp1d(df_results['public_burden_b'], df_results['combined_pct'], 
                    kind='linear', fill_value='extrapolate')

# Calculate needed building code levels
bc_needed_econ = f_econ(baseline_economic)
bc_needed_defaults = f_defaults(baseline_defaults)
bc_needed_public = f_public(baseline_public)

print("Building code levels needed to return to ERA5 baseline:")
print(f"  Economic Burden: {bc_needed_econ:.1f}% combined wind+flood reduction")
print(f"  Defaults: {bc_needed_defaults:.1f}% combined wind+flood reduction")
print(f"  Public Burden: {bc_needed_public:.1f}% combined wind+flood reduction")

# Calculate uncertainty bounds on needed levels using P10/P90
f_econ_p10 = interp1d(df_results['economic_burden_b_p10'], df_results['combined_pct'], 
                       kind='linear', fill_value='extrapolate')
f_econ_p90 = interp1d(df_results['economic_burden_b_p90'], df_results['combined_pct'], 
                       kind='linear', fill_value='extrapolate')

bc_needed_econ_p10 = f_econ_p10(baseline_economic)
bc_needed_econ_p90 = f_econ_p90(baseline_economic)

print(f"\nEconomic Burden uncertainty range: {bc_needed_econ_p10:.1f}% - {bc_needed_econ_p90:.1f}%")

## 7. Summary Statistics

Compute key statistics across all building code levels.

In [ ]:
# Summary table
summary_data = []

for bc_label, (wind_pct, flood_pct) in building_code_levels.items():
    if bc_label in era5_climate_adjusted:
        metrics = era5_climate_adjusted[bc_label]
        combined_pct = (wind_pct + flood_pct) / 2
        
        summary_data.append({
            'Building Code': bc_label,
            'Wind Reduction': f'{wind_pct}%',
            'Flood Reduction': f'{flood_pct}%',
            'Combined': f'{combined_pct:.1f}%',
            'Economic Burden ($B)': f"{metrics['total_economic_burden']/1e9:.2f}",
            'Defaults': f"{metrics['defaults']:.2f}",
            'Public Burden ($B)': f"{metrics['total_public_burden']/1e9:.2f}",
        })

df_summary = pd.DataFrame(summary_data)
print("\nSummary Table: Climate-Adjusted ERA5 Results (SSP245 2050)")
print("=" * 100)
print(df_summary.to_string(index=False))
print("=" * 100)
print(f"\nCurrent Climate Baseline (ERA5):")
print(f"  Economic Burden: ${baseline_economic:.2f}B")
print(f"  Defaults: {baseline_defaults:.2f}")
print(f"  Public Burden: ${baseline_public:.2f}B")

## 8. Export Results

Save results for use in other notebooks/reports.

In [ ]:
# Export to CSV
df_results.to_csv('../results/climate_buildingcode_windfloods_results.csv', index=False)
print("✓ Results exported to: ../results/climate_buildingcode_windfloods_results.csv")